# Sys1Pop Studio: Frontier LLM Teacher to Edge Distillation

This notebook demonstrates how to use a frontier LLM (Claude, Gemini, GPT) as a **Teacher** to synthesize domain training data, train a lightweight 33.4M parameter edge model in under 60 seconds on CPU, quantize it to INT8, and export the **Sys1Pop Model Bundle** ready for Cloudflare Workers.

In [ ]:
# Step 1: Install lightweight dependencies
!pip install -q safetensors sentence-transformers tokenizers

In [ ]:
# Step 2: Define your domain questions & schema
model_id = "healthcare-triage-v1"

manifest = {
    "schema_version": "1.0",
    "model_id": model_id,
    "architecture": "minilm_l6_v2",
    "hidden_dim": 384,
    "max_seq_len": 512,
    "quantization": "int8_q8_0",
    "supported_heads": ["choice", "boolean", "score"],
    "default_questions": [
        {"type": "boolean", "id": "hipaa_risk"},
        {"type": "choice", "id": "department", "options": ["clinical", "billing", "pharmacy", "emergency"]},
        {"type": "score", "id": "triage_urgency", "min": 1, "max": 5}
    ],
    "calibration": {"temperature": 1.0, "default_threshold": 0.5}
}
print(f"Defined schema for {model_id}")

In [ ]:
# Step 3: Synthetic data generated by Frontier LLM
synthetic_dataset = [
    {"text": "Patient SSN 000-00-0000 reported severe chest pain radiating to left arm.", "hipaa_risk": True, "department": "emergency", "triage_urgency": 5},
    {"text": "Need to update my health insurance copay card before next Tuesday.", "hipaa_risk": False, "department": "billing", "triage_urgency": 2},
    {"text": "Doctor prescribed amoxicillin 500mg, need a refill sent to CVS pharmacy.", "hipaa_risk": False, "department": "pharmacy", "triage_urgency": 3},
    {"text": "Routine appointment request for annual dermatology skin check.", "hipaa_risk": False, "department": "clinical", "triage_urgency": 1}
]
print(f"Loaded {len(synthetic_dataset)} synthetic training exemplars")

In [ ]:
# Step 4: Export Sys1Pop Bundle using tools/distill_and_export.py
import os
!python ../tools/distill_and_export.py --model-id {model_id} --output-dir ./dist/{model_id}

print("\nBundle generated successfully! Ready for edge publishing:")
print(f"npx sys1pop model push ./dist/{model_id} --name {model_id}")